In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from scripts import convert_dates, query_db

# Set pandas option to display all columns
pd.set_option('display.max_columns', None)

In [ ]:
# Read the list of tables that exist in the database
tables = query_db("SELECT name FROM sqlite_master WHERE type='table'")

In [ ]:
# Query schemas of each table
table_schemas = dict()
for table_name in tables['name']:
    table_schemas[table_name] = query_db(f"PRAGMA table_info({table_name});")
    print(f"Schema for table '{table_name}':")
    print(table_schemas[table_name])
    print("\n")

In [ ]:
matches = query_db("SELECT * FROM MatchResult")
levels = query_db("SELECT * FROM LevelResult")
player_matches = query_db("SELECT * FROM PlayerMatchResult pmr LEFT JOIN MatchResult mr ON pmr.MatchResultId = mr.MatchResultId")
player_damage = query_db(Path("player-damage.sql").read_text())
player_deaths = query_db("SELECT * FROM PlayerDamage WHERE IsFatal = 1")
player_spawns = query_db(Path("player-spawns.sql").read_text())
player_levels = query_db(Path("player-level-results.sql").read_text())
player_lives = query_db(Path("player-lives.sql").read_text())
powerups = query_db(Path("powerups.sql").read_text())

In [ ]:
player_damage["IsLevelObject"] = ~player_damage.SourceType.isin(("Player", "Bullet", "Stunbullet"))
has_level_obj_damage = (
    player_damage
    [["MatchResultId", "LevelResultId", "IsLevelObject"]]
    .groupby(["MatchResultId", "LevelResultId"])
    .any().reset_index()
    .rename(columns={"IsLevelObject": "LevelObjectDamageOccurred"})
)

In [ ]:
powerups["HasPowerupPickup"] = True
has_powerup_pickup = (
    powerups
    [["MatchResultId", "LevelResultId", "HasPowerupPickup"]]
    .groupby(["MatchResultId", "LevelResultId"])
    .any().reset_index()
)

In [ ]:
levels = levels.merge(
    has_level_obj_damage,
    on=["MatchResultId", "LevelResultId"],
    how="left",
).merge(
    has_powerup_pickup,
    on=["MatchResultId", "LevelResultId"],
    how="left",
)
levels["HasPowerupPickup"] = levels.HasPowerupPickup.fillna(False, inplace=True)

In [ ]:
inferrences = query_db(Path("infer-levels.sql").read_text())
inferrences = inferrences.merge(
    levels[["MatchResultId", "LevelResultId", "StartTime", "HasPowerupPickup", "LevelObjectDamageOccurred"]],
    on=["MatchResultId", "LevelResultId"],
    how="left",
)
inferrences

In [ ]:
inferrences["SpawnSecondsIntoLevel"] = (
    inferrences.SpawnTime - inferrences.StartTime
).dt.total_seconds()

In [ ]:
spawn_point_comparisons = inferrences[inferrences.LevelName == "barrel"][[
    "MatchResultId",
    "LevelResultId",
    "SpawnId",
    "SpawnSecondsIntoLevel",
    "SpawnX",
    "SpawnY",
    "InferredLevelName",
    "InferredSpawnPoint",
    "InferredSpawnX",
    "InferredSpawnY",
    "Distance",
]].rename(columns={
    "InferredLevelName": "ComparisonLevelName",
    "InferredSpawnPoint": "ComparisonSpawnPoint",
    "InferredSpawnX": "ComparisonSpawnX",
    "InferredSpawnY": "ComparisonSpawnY",
}).copy()
display(spawn_point_comparisons.info())
df = spawn_point_comparisons

In [ ]:
import pandas as pd

best_matches = (
    df
    .groupby(
        [
            "MatchResultId",
            "LevelResultId",
            "SpawnId",
            "ComparisonLevelName",
            "SpawnSecondsIntoLevel",
        ],
        as_index=False
    )["Distance"]
    .min()
)


In [ ]:
# Choose bin size (seconds)
TIME_BIN = 5  # adjust as needed

best_matches["TimeBin"] = (
    best_matches["SpawnSecondsIntoLevel"] // TIME_BIN
) * TIME_BIN


In [ ]:
time_series_metrics = (
    best_matches
    .groupby(["ComparisonLevelName", "TimeBin"])
    .agg(
        total_distance=("Distance", "sum"),
        avg_distance=("Distance", "mean"),
        spawn_count=("Distance", "size"),
    )
    .reset_index()
)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

for level, g in time_series_metrics.groupby("ComparisonLevelName"):
    plt.plot(
        g["TimeBin"],
        g["total_distance"],
        label=level,
        alpha=0.7,
    )

plt.xlabel("Seconds Into Level (binned)")
plt.ylabel("Total Distance")
plt.title("Total Spawn Distance vs Time by Candidate Level")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))

for level, g in time_series_metrics.groupby("ComparisonLevelName"):
    plt.plot(
        g["TimeBin"],
        g["avg_distance"],
        label=level,
        alpha=0.7,
    )

plt.xlabel("Seconds Into Level (binned)")
plt.ylabel("Average Distance")
plt.title("Average Spawn Distance vs Time by Candidate Level")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
time_series_metrics["distance_per_spawn"] = (
    time_series_metrics["total_distance"]
    / time_series_metrics["spawn_count"]
)


In [ ]:
incorrect_inferrences = inferrences[(inferrences.InferredSpawnPoint == 1) & (inferrences.IsInferredLevel == 1) & (inferrences.IsMultiLevelMatch == 0) & (inferrences.IsCorrectInference == 0)]
incorrectly_inferred_levels = incorrect_inferrences.LevelResultId.unique()

In [ ]:
incorrect_inferrences.LevelName.value_counts()

In [ ]:
inferrences[inferrences.LevelResultId.isin(incorrectly_inferred_levels) & (inferrences.LevelResultId == 103)].to_csv("incorrect_inferences_levelresultid.csv", index=False)

In [ ]:
levels[~(levels.HasPowerupPickup | levels.LevelObjectDamageOccurred | (levels.LevelName == "barrel"))].merge(
    player_levels, on=["MatchResultId", "LevelResultId"], how="left", suffixes=("", "_")
).LevelName.value_counts()

In [ ]:
(
    inferrences
    [inferrences.IsBestPoint == 1]
    [["MatchResultId", "LevelResultId", "InferredLevelName", "Distance"]]
    .groupby(["MatchResultId", "LevelResultId", "InferredLevelName"])
    .sum().reset_index()
)

In [ ]:
levels

In [ ]:
correct_distances = (
    inferrences
    [(inferrences.IsMultiLevelMatch == 0) & (inferrences.IsBestPoint == 1) & (inferrences.IsTrueLevel == 1)]
    [["MatchResultId", "LevelResultId", "LevelName", "Distance"]]
    .groupby(["MatchResultId", "LevelResultId", "LevelName"])
    .sum().reset_index()
    .rename(columns={"Distance": "TotalDistance"})
)
correct_distance_averages = (
    inferrences
    [(inferrences.IsMultiLevelMatch == 0) & (inferrences.IsBestPoint == 1) & (inferrences.IsTrueLevel == 1)]
    [["MatchResultId", "LevelResultId", "LevelName", "Distance"]]
    .groupby(["MatchResultId", "LevelResultId", "LevelName"])
    .mean().reset_index()
    .rename(columns={"Distance": "AverageDistance"})
)
correct_distances = correct_distances.merge(
    correct_distance_averages, on=["MatchResultId", "LevelResultId", "LevelName"], how="left"
)
correct_distances.sort_values("TotalDistance", ascending=False).merge(levels[["MatchResultId", "LevelResultId", "LevelObjectDamageOccurred", "HasPowerupPickup"]], on=["MatchResultId", "LevelResultId"], how="left").to_csv("correct_inferences_distances.csv", index=False)

In [ ]:
inferrences[inferrences.IsInferredLevel == 1]

In [ ]:
matches.LevelCount.value_counts()

In [ ]:
from collections import defaultdict
level_spawns = player_spawns[player_spawns.LevelCount == 1][["LevelResultId", "SpawnId", "X", "Y", "LevelName"]].copy()

# Infer level based on spawn point proximity
# Group all spawns by LevelResultId and check which set of 4 spawn points best matches the spawn pattern observed for the LevelResultId
# There are multiple spawns at each spawn point per LevelResultId, so we have to first identify which spawn points in various levels seem to match
# Then we can see which level best describes the set of all spawns for that LevelResultId

# For each SpawnId, track the inferred spawn point and level
inferred_levels_df = pd.DataFrame()
spawn_inferred = {}
for level_result_id, group in level_spawns.groupby("LevelResultId"):
    # For each spawn in the group, find the closest spawn point in each level
    # Track the level, distance, and spawn point ID
    # Orient the data around the levels so we can later aggregate per level
    best_point_per_spawn_per_level = defaultdict(list)
    for spawn in group.itertuples():
        spawn_id = spawn.SpawnId
        true_level_name = spawn.LevelName
        spawn_x = spawn.X
        spawn_y = spawn.Y

        for level_name, points in spawn_points.items():
            best_distance = float('inf')
            best_spawn_point = None
            best_spawn_xy = None
            for point_id, point in enumerate(points):
                point_x, point_y = point
                distance = (((spawn_x - point_x) ** 2) + ((spawn_y - point_y) ** 2)) ** 0.5
                if distance < best_distance:
                    best_distance = distance
                    best_spawn_point = point_id
                    best_spawn_xy = point

            # Record the best match for this spawn in this level
            best_point_per_spawn_per_level[level_name].append((
                level_result_id,
                spawn_id,
                best_distance,
                best_spawn_point,
                best_spawn_xy,
                level_name == true_level_name,
            ))

    # Now we have each level associated with the best spawn point for all spawns in this LevelResultId
    # we can aggregate to see which level has the lowest total distance
    level_scores = {}
    true_level_scores = list()  # also track the score of the known true level so we can see how well we did
    for level_name, spawn_matches in best_point_per_spawn_per_level.items():
        total_distance = sum([match[2] for match in spawn_matches])
        level_scores[level_name] = (total_distance, spawn_matches)

        # make sure we can associate the true level score with the SpawnId and LevelResultId
        if spawn_matches[0][5]:  # if this level is the true level
            for match in spawn_matches:
                true_level_scores.append((match[0], match[1], total_distance, level_name))

    # Find the level with the lowest total distance
    best_level = min(level_scores.items(), key=lambda x: x[1][0])
    inferred_level_name = best_level[0]
    inferred_score = best_level[1][0]

    inferred_level_df = pd.DataFrame(best_level[1][1], columns=["LevelResultId", "SpawnId", "Distance", "InferredSpawnPoint", "InferredSpawnXY", "IsTrueLevel"])
    inferred_level_df["InferredLevelName"] = inferred_level_name
    inferred_level_df["InferenceScore"] = inferred_score
    inferred_level_df["InferredSpawnX"] = inferred_level_df.InferredSpawnXY.map(lambda x: x[0])
    inferred_level_df["InferredSpawnY"] = inferred_level_df.InferredSpawnXY.map(lambda x: x[1])

    inferred_levels_df = pd.concat([inferred_levels_df, inferred_level_df], ignore_index=True)
    true_level_df = pd.DataFrame(true_level_scores, columns=["LevelResultId", "SpawnId", "TrueLevelScore", "TrueLevelName"])



In [ ]:
inf = (
    level_spawns.merge(
        inferred_levels_df,
        on=["LevelResultId", "SpawnId"],
        suffixes=("_Actual", "_Inferred"),
        how="left"
    ).merge(
        true_level_df,
        on=["LevelResultId", "SpawnId"],
        how="left"
    )
)
inf

In [ ]:
inf[inf.LevelName != inf.InferredLevelName]

In [ ]:
player_damage[(player_damage.LevelName == "valley") & (~player_damage.SourceType.isin(("Player", "HurtPoint", "SeaMineExplosion", "Pelican", "Rock")) & (~player_damage.SourceType.str.contains("KillBox").fillna(False)))]

In [ ]:
levels["LevelDuration"] = pd.to_datetime(levels["EndTime"]) - pd.to_datetime(levels["StartTime"])
levels["LevelDurationSeconds"] = levels["LevelDuration"].dt.total_seconds()
levels

In [ ]:
print(player_damage[player_damage.SourceType != "Player"][["LevelName", "SourceType"]].value_counts().reset_index().to_markdown())

In [ ]:
player_matches

In [ ]:
## Span point XY locations per level
spawn_points = {
    "acid_factory": [
        (-23.63516, -8.419025),
        (5.274841, -8.399027),
        (-15.52516, -8.449028),
        (-4.605159, -8.419025),
    ],
    "barrel": [
        (-6.3, 0.9),
        (3.7, 2.2),
        (1.6, -9.1),
        (-4.78, -9.1),
    ],
    "beach": [
        (-12.24, -1.54),
        (-4.86, -1.77),
        (6.74, -4.11),
        (13.33, 1.01),
    ],
    "cargo_hold": [
        (0, 9.5),
        (7.5, 2.5),
        (-7.5, 2.5),
        (0, -5),
    ],
    "firing_range": [
        (-6.82, -4.14),
        (-7.87, -9.2),
        (9.88, -3.8),
        (21.49, -11.84),
    ],
    "valley": [
        (9.36, -14.22),
        (-1.87, -17.33),
        (-7.08, -24.84),
        (4.45, -23.13),
    ]
}

In [ ]:
# Identify how closely the pattern of spawns in each level resembles the spawns in every other level
# For each spawn point in each level, find the closest spawn point in every other level
# Then sum the distances for each level to get a score for how closely the spawn patterns match
for level_name, points in spawn_points.items():
    print(f"Level: {level_name}")
    for other_level_name, other_points in spawn_points.items():
        if level_name == other_level_name:
            continue
        total_distance = 0
        for spawn_point in points:
            # Find the closest spawn point in the other level
            distances = [((spawn_point[0] - other_spawn_point[0]) ** 2 + (spawn_point[1] - other_spawn_point[1]) ** 2) ** 0.5 for other_spawn_point in other_points]
            min_distance = min(distances)
            total_distance += min_distance
        print(f"  Closest to level {other_level_name} with total distance {total_distance:.2f}")
    print("\n")

# GunFish Stats 2026!
Data analysis for GunFish in MAGFest's Indie Arcade 2026.

## Basics
How did players approach the game this year?

Some definitions:
* "Matches" - a play session consisting either of the single-level deathmatch or the three-level deathmatch
* "Levels" - individual level plays
* "Plays" - essentially matches times the player count of the match

In [ ]:
pd.DataFrame({
    "Total Matches": [matches.MatchResultId.nunique()],
    "Total Levels": [levels.LevelResultId.nunique()],
    "Total Plays": [player_matches.PlayerMatchResultId.nunique()],
    "Total Damage Events": [player_damage.DamageId.nunique()],
    "Total Deaths": [player_deaths.DamageId.nunique()],
    "Non-Scoring Plays": [(player_matches.Score == 0).sum()],
    "Scoring Percentage": [round(100 * (player_matches.Score > 0).sum() / len(player_matches), 1)],
    "Highest Level Kill Count": [player_levels.Kills.max()],
    "Highest One-Level Score": [player_matches[player].Score.max()],
})

In [ ]:
# Display pie chart of match player counts
match_player_counts = matches.PlayerCount.value_counts().sort_index()
match_player_counts.plot(kind="pie", autopct="%1.1f%%", startangle=90)
plt.title("Match Player Counts")
plt.show()

In [ ]:
# Display pie chart of level player counts

# Merge level results with match results to get player counts per level
level_matches = pd.merge(
    levels,
    matches[['MatchResultId', 'PlayerCount']],
    on='MatchResultId',
    how='left'
)
level_counts = level_matches.PlayerCount.value_counts().sort_index()
level_counts.plot(kind="pie", autopct="%1.1f%%", startangle=90)
plt.title("Level Player Counts")
plt.show()

In [ ]:
# Display stats in dataframe of 3-level vs. 1-level matches by player count using MatchResult.PLayerCount and MatchResult.LevelCount
# show the counts and also indicate what percent of matches are 3-level vs 1-level for each player count
level_count_stats = matches.groupby('PlayerCount')['LevelCount'].value_counts(normalize=True).unstack(fill_value=0) * 100
level_count_stats = level_count_stats.rename(columns={1: '1-Level (%)', 3: '3-Level (%)'})
print(level_count_stats.to_markdown())

In [ ]:
# Display pie chart of 1-level vs 3-level matches
level_counts = matches['LevelCount'].value_counts().sort_index()
level_counts.plot(kind="pie", autopct="%1.1f%%", startangle=90)
plt.title("1-Level vs 3-Level Matches")
plt.show()

In [ ]:
# Display a bar chart of average kills per fish life
avg_kills_per_life = player_lives.groupby("PlayerFish").Kills.mean().sort_values()
avg_kills_per_life.plot(kind="bar")
plt.title("Average Kills per Fish Life")
plt.xlabel("Fish Life")
plt.ylabel("Average Kills")
plt.show()

In [ ]:
# Display a bar chart of average kills per fish per match
avg_kills_per_fish_match = player_lives.groupby(["PlayerFish", "MatchResultId"]).Kills.sum().groupby("PlayerFish").mean().sort_values()
avg_kills_per_fish_match.plot(kind="bar")
plt.title("Average Kills per Fish per Match")
plt.xlabel("Fish Life")
plt.ylabel("Average Kills per Match")
plt.show()

In [ ]:
# Display a bar chart of average damage dealt per fish life
avg_damage_per_life = player_lives.groupby("PlayerFish").TotalDamageDealt.mean().sort_values()
avg_damage_per_life.plot(kind="bar")
plt.title("Average Damage Dealt per Fish Life")
plt.xlabel("Fish Life")
plt.ylabel("Average Damage Dealt")
plt.show()

In [ ]:
# Display bar chart of average life duration per fish life
avg_life_duration_per_fish = player_lives.groupby("PlayerFish").LifeDuration.mean().sort_values()
# convert to seconds for easier reading
avg_life_duration_per_fish = avg_life_duration_per_fish.dt.total_seconds()
avg_life_duration_per_fish.plot(kind="bar")
plt.title("Average Life Seconds per Fish Life")
plt.xlabel("Fish Life")
plt.ylabel("Average Life Seconds")
plt.show()

In [ ]:
# Display notched box chart of average life duration per fish type in seconds
avg_life_duration_per_fish_type = player_lives.groupby("PlayerFish").LifeDuration.mean().sort_values()
# convert to seconds for easier reading
avg_life_duration_per_fish_type = avg_life_duration_per_fish_type.dt.total_seconds()
avg_life_duration_per_fish_type.plot(kind="box", notch=True)
plt.title("Average Life Duration per Fish Type")
plt.show()